<a href="https://colab.research.google.com/github/leovarconrenno/Estacao-de-Reabastecimento-de-Hidrogenio---SCADA-Core/blob/main/etapa-02-grafos/14%20-%20Menor%20Caminho%20Dijkstra%20e%20Roteamento%20Dinamico.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 14 - Notebook: Algoritmo de Dijkstra e Roteamento Dinâmico de Pressão (SCADA H₂)

Neste notebook implementamos o **Algoritmo de Dijkstra** com fila de prioridade binária (`heapq`) para encontrar a rota de menor comprimento total de tubulação entre `SUP-100` e `VEIC-300`, comparando as três estratégias de cascata (LP, MP e HP) da Estação de Reabastecimento de Hidrogênio.

![Grafo da Estação de Hidrogênio](grafo_tubulacoes.png)

In [1]:
import sys
sys.stdout.reconfigure(encoding='utf-8')

def formatar_tabela(dados):
    # Formata lista de dicionarios em tabela ASCII pura
    if not dados:
        return 'Tabela Vazia'
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ''))))
    header = ' | '.join(f'{c:<{larguras[c]}}' for c in colunas)
    divisor = '-+-'.join('-' * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(' | '.join(f'{str(row.get(c, "")):<{larguras[c]}}' for c in colunas))
    return '\n'.join(linhas)


class GrafoTubulacao:
    def __init__(self, vertices):
        self.vertices = vertices
        self.v_to_idx = {v: i for i, v in enumerate(vertices)}
        self.idx_to_v = {i: v for i, v in enumerate(vertices)}
        self.n = len(vertices)
        self.adj_binaria = [[0] * self.n for _ in range(self.n)]
        self.adj_pesos = [[float('inf')] * self.n for _ in range(self.n)]
        for i in range(self.n):
            self.adj_pesos[i][i] = 0.0
        self.arestas_detalhes = []

    def adicionar_tubulacao(self, origem, destino, comprimento_m, tag_valvula, diametro_pol=1.0):
        u = self.v_to_idx[origem]
        v = self.v_to_idx[destino]
        self.adj_binaria[u][v] = 1
        self.adj_pesos[u][v] = comprimento_m
        self.arestas_detalhes.append({
            'Origem': origem, 'Destino': destino,
            'Comprimento (m)': comprimento_m,
            'Valvula ISA': tag_valvula,
            'Diametro (pol)': diametro_pol
        })


def criar_rede_h2():
    nos = ['SUP-100', 'TK-101', 'TK-102', 'TK-103', 'XV-104', 'HEX-201', 'DISP-300', 'VEIC-300']
    g = GrafoTubulacao(nos)
    g.adicionar_tubulacao('SUP-100', 'TK-101',  10.0, 'XV-101', 1.0)
    g.adicionar_tubulacao('SUP-100', 'TK-102',  12.0, 'XV-102', 1.0)
    g.adicionar_tubulacao('SUP-100', 'TK-103',  15.0, 'XV-103', 0.5)
    g.adicionar_tubulacao('TK-101',  'XV-104',   8.0, 'Linha LP', 1.0)
    g.adicionar_tubulacao('TK-102',  'XV-104',   6.0, 'Linha MP', 1.0)
    g.adicionar_tubulacao('TK-103',  'XV-104',   8.0, 'Linha HP', 0.5)
    g.adicionar_tubulacao('XV-104',   'HEX-201',  14.0, 'XV-201', 1.0)
    g.adicionar_tubulacao('HEX-201',  'DISP-300', 16.0, 'XV-301', 0.5)
    g.adicionar_tubulacao('DISP-300', 'VEIC-300',  4.0, 'BV-301', 0.5)
    return g


import heapq
from typing import List, Tuple, Optional, Set

rede = criar_rede_h2()


class RoteadorDijkstra:
    def __init__(self, grafo: GrafoTubulacao):
        self.g = grafo

    def calcular_menor_caminho(
        self, origem: str, destino: str,
        bloqueios: Optional[Set[str]] = None
    ) -> Tuple[float, List[str]]:
        if bloqueios is None:
            bloqueios = set()
        if origem in bloqueios or destino in bloqueios:
            return float('inf'), []
        dist = {v: float('inf') for v in self.g.vertices}
        pred = {v: None for v in self.g.vertices}
        dist[origem] = 0.0
        heap = [(0.0, origem)]
        while heap:
            d_u, u = heapq.heappop(heap)
            if d_u > dist[u]:
                continue
            if u == destino:
                break
            u_idx = self.g.v_to_idx[u]
            for v_idx in range(self.g.n):
                v = self.g.idx_to_v[v_idx]
                peso = self.g.adj_pesos[u_idx][v_idx]
                if peso < float('inf') and v not in bloqueios:
                    nova_d = d_u + peso
                    if nova_d < dist[v]:
                        dist[v] = nova_d
                        pred[v] = u
                        heapq.heappush(heap, (nova_d, v))
        caminho = []
        atual = destino
        while atual is not None:
            caminho.append(atual)
            atual = pred[atual]
        caminho.reverse()
        if caminho and caminho[0] == origem:
            return dist[destino], caminho
        return float('inf'), []


roteador = RoteadorDijkstra(rede)

# Rota nominal otima -- Cascata LP
custo_nom, rota_nom = roteador.calcular_menor_caminho('SUP-100', 'VEIC-300')
print(f'Rota Nominal Otima (LP): {" -> ".join(rota_nom)} | Comprimento: {custo_nom:.1f} m')
assert custo_nom == 52.0, f'Esperado 52.0 m, obtido {custo_nom}'

# Rota alternativa via TK-102 -- Cascata MP
custo_mp, rota_mp = roteador.calcular_menor_caminho('SUP-100', 'VEIC-300', bloqueios={'TK-101'})
print(f'Rota Alternativa MP:    {" -> ".join(rota_mp)} | Comprimento: {custo_mp:.1f} m')
assert custo_mp == 52.0

# Rota alternativa via TK-103 -- Cascata HP
custo_hp, rota_hp = roteador.calcular_menor_caminho('SUP-100', 'VEIC-300', bloqueios={'TK-101', 'TK-102'})
print(f'Rota Alternativa HP:    {" -> ".join(rota_hp)} | Comprimento: {custo_hp:.1f} m')
assert custo_hp == 57.0

print()
print('--- Comparativo de Rotas de Abastecimento (Dijkstra) ---')
relatorio = [
    {'Estagio': 'LP (TK-101, 350-400 bar)',  'Rota': ' -> '.join(rota_nom),
     'Comprimento_m': f'{custo_nom:.1f}', 'Status': 'ROTA OTIMA'},
    {'Estagio': 'MP (TK-102, 650-700 bar)',  'Rota': ' -> '.join(rota_mp),
     'Comprimento_m': f'{custo_mp:.1f}',  'Status': 'Contingencia 1'},
    {'Estagio': 'HP (TK-103, 950-1000 bar)', 'Rota': ' -> '.join(rota_hp),
     'Comprimento_m': f'{custo_hp:.1f}',  'Status': 'Contingencia 2'},
]
print(formatar_tabela(relatorio))


Rota Nominal Otima (LP): SUP-100 -> TK-101 -> XV-104 -> HEX-201 -> DISP-300 -> VEIC-300 | Comprimento: 52.0 m
Rota Alternativa MP:    SUP-100 -> TK-102 -> XV-104 -> HEX-201 -> DISP-300 -> VEIC-300 | Comprimento: 52.0 m
Rota Alternativa HP:    SUP-100 -> TK-103 -> XV-104 -> HEX-201 -> DISP-300 -> VEIC-300 | Comprimento: 57.0 m

--- Comparativo de Rotas de Abastecimento (Dijkstra) ---
Estagio                   | Rota                                                                   | Comprimento_m | Status        
--------------------------+------------------------------------------------------------------------+---------------+---------------
LP (TK-101, 350-400 bar)  | SUP-100 -> TK-101 -> XV-104 -> HEX-201 -> DISP-300 -> VEIC-300       | 52.0          | ROTA OTIMA    
MP (TK-102, 650-700 bar)  | SUP-100 -> TK-102 -> XV-104 -> HEX-201 -> DISP-300 -> VEIC-300       | 52.0          | Contingencia 1
HP (TK-103, 950-1000 bar) | SUP-100 -> TK-103 -> XV-104 -> HEX-201 -> DISP-300 -> VEIC-300